# 🏦 Banking NLP — Loan Approval Sentiment Analyser
### 📝 Learner Worksheet

---

## 📌 Project Overview

When a bank approves or rejects a loan, customers leave feedback.  
Your job is to analyse that feedback using NLP to answer:

- **How do customers FEEL about their loan decision?**
- **What words do they use most?**

## 📋 Your 6 Tasks

| Task | What You Will Do | Concept |
|------|-----------------|----------|
| Task 1 | Load and explore the dataset | Pandas, JSON |
| Task 2 | Clean raw text | Regex |
| Task 3 | Tokenize and filter words | Basic NLP — NLTK |
| Task 4 | Score sentiment | VADER + TextBlob |
| Task 5 | Classify into 4 business groups | Logic + Sentiment |
| Task 6 | Visualise the results | Matplotlib + WordCloud |

---

### 📖 How to use this worksheet
- Each task has an explanation of **what to do** and **why**
- Code cells have **blanks marked with `____`** for you to fill in
- **Hints** are provided inside comments
- Run each cell after completing it to check your work

---

## ⚙️ Setup — Install and Import Libraries

**Run this cell first — do not change anything here.**

In [ ]:
!pip install vaderSentiment textblob wordcloud nltk pandas matplotlib -q

import nltk
nltk.download('punkt', quiet=True)
nltk.download('punkt_tab', quiet=True)
nltk.download('stopwords', quiet=True)

print("✅ All libraries ready!")

In [ ]:
import re
import json
import pandas as pd
import matplotlib.pyplot as plt

from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
from nltk.probability import FreqDist

from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer
from textblob import TextBlob
from wordcloud import WordCloud

import warnings
warnings.filterwarnings('ignore')

print("✅ Imports done!")

---
## 📋 Task 1 — Load and Explore the Dataset

### What to do
The customer feedback data is stored as **JSON** (the format real APIs return).  
You need to:
1. Parse the JSON string into a Python list using `json.loads()`
2. Convert that list into a Pandas DataFrame
3. Explore the data using `.head()` and `.info()`

### Key concept — JSON to DataFrame
```
JSON string  →  json.loads()  →  Python list of dicts  →  pd.DataFrame()  →  Table
```

### Dataset columns
| Column | Meaning |
|---|---|
| `customer_id` | Unique customer ID |
| `loan_status` | Approved or Rejected |
| `feedback` | What the customer said |

In [ ]:
# ── The dataset is given to you — do not change this ─────────────
api_response = json.dumps([
    {"customer_id": "C001", "loan_status": "Approved",  "feedback": "I am so happy my loan got approved! The process was smooth and the staff were very helpful."},
    {"customer_id": "C002", "loan_status": "Rejected",  "feedback": "Very disappointed. My loan was rejected without a clear reason. Terrible service."},
    {"customer_id": "C003", "loan_status": "Approved",  "feedback": "Great news! Loan approved quickly. Excellent support from the bank team."},
    {"customer_id": "C004", "loan_status": "Rejected",  "feedback": "Unfair decision. I submitted all documents but still got rejected. Very frustrated."},
    {"customer_id": "C005", "loan_status": "Approved",  "feedback": "Wonderful experience. Got approved within 2 days. Will recommend to friends."},
    {"customer_id": "C006", "loan_status": "Approved",  "feedback": "The approval was fast but the interest rate is too high. Not entirely satisfied."},
    {"customer_id": "C007", "loan_status": "Rejected",  "feedback": "I understand the rejection but wish the bank had explained the reason better."},
    {"customer_id": "C008", "loan_status": "Approved",  "feedback": "Happy with the approval. The EMI plan is flexible and manageable."},
    {"customer_id": "C009", "loan_status": "Rejected",  "feedback": "Shocking rejection after waiting 3 weeks. Waste of time and very unprofessional."},
    {"customer_id": "C010", "loan_status": "Approved",  "feedback": "Finally approved after two attempts. The second time the process was much clearer."},
    {"customer_id": "C011", "loan_status": "Rejected",  "feedback": "Rejection was expected but the communication was poor. No follow up call received."},
    {"customer_id": "C012", "loan_status": "Approved",  "feedback": "Brilliant service. Loan approved faster than expected. Very professional team."},
    {"customer_id": "C013", "loan_status": "Rejected",  "feedback": "Deeply upset about the rejection. I needed this loan urgently for medical expenses."},
    {"customer_id": "C014", "loan_status": "Approved",  "feedback": "Approval received but the paperwork was excessive and confusing for a first-time applicant."},
    {"customer_id": "C015", "loan_status": "Rejected",  "feedback": "The rejection letter was polite and the bank suggested ways to improve my application."},
    {"customer_id": "C016", "loan_status": "Approved",  "feedback": "Very satisfied. Interest rate is fair and the repayment schedule suits my income."},
    {"customer_id": "C017", "loan_status": "Rejected",  "feedback": "Frustrated with the outcome. Applied twice and got rejected both times with no explanation."},
    {"customer_id": "C018", "loan_status": "Approved",  "feedback": "Good experience overall. Minor delays in documentation but final approval was smooth."},
    {"customer_id": "C019", "loan_status": "Rejected",  "feedback": "Neutral about it. The rejection was fair given my current credit score situation."},
    {"customer_id": "C020", "loan_status": "Approved",  "feedback": "Thrilled with the quick approval. The bank manager personally called to congratulate me."}
])

# ── YOUR TASK ────────────────────────────────────────────────────
# Step 1: Parse the JSON string into a Python list
# Hint: use json.loads(api_response)
data = ____

# Step 2: Convert the list into a Pandas DataFrame
# Hint: use pd.DataFrame(data)
df = ____

# Step 3: Print how many rows and the loan status counts
# Hint: len(df) gives number of rows
print(f"Total records: {____}")
print(f"Approved: {(df['loan_status']=='Approved').sum()}")
print(f"Rejected: {(df['loan_status']=='Rejected').sum()}")

In [ ]:
# ── YOUR TASK ────────────────────────────────────────────────────
# Show the first 5 rows of the DataFrame
# Hint: use df.head(____)
____

---
## 🔧 Task 2 — Clean Text Using Regex

### What to do
Customer feedback contains punctuation, numbers, and symbols that interfere with NLP.  
Write a `clean_text()` function using regex to:
1. Remove anything that is NOT a letter or space
2. Collapse multiple spaces into one
3. Strip whitespace from ends and convert to lowercase

### Regex patterns to use
| Pattern | What it does |
|---|---|
| `[^a-zA-Z\s]` | Matches anything that is NOT a letter or space |
| `\s+` | Matches one or more whitespace characters |

### Expected output
```
Input:  "I am so happy! Loan approved in 2 days."
Output: "i am so happy loan approved in days"
```

---
## 🧠 Task 3 — Tokenize and Remove Stopwords

### What to do
Break each cleaned sentence into individual words, then remove common words that carry no meaning.

### Steps inside `tokenize_and_filter()`
1. `word_tokenize(text)` — splits sentence into word list
2. Keep only alphabetic tokens — `token.isalpha()`
3. Remove stopwords — check if token is NOT in `STOP_WORDS`
4. Remove very short tokens — keep only words longer than 2 characters

### After tokenizing — use FreqDist
Flatten all token lists into one big list, then count word frequencies.

### Expected output (example)
```
Input : "i am so happy loan approved quickly"
Tokens: ['happy', 'approved', 'quickly']
```

---
## 💬 Task 4 — Sentiment Scoring with VADER and TextBlob

### What to do
Score each customer's feedback using both VADER and TextBlob, then build a **consensus label**.

### VADER rules
- `compound >= 0.05` → `'Positive'`
- `compound <= -0.05` → `'Negative'`
- Otherwise → `'Neutral'`
- Feed VADER the **original `feedback`** column (not cleaned text)

### TextBlob rules
- `polarity > 0.05` → `'Positive'`
- `polarity < -0.05` → `'Negative'`
- Otherwise → `'Neutral'`

### Consensus rule
- Both agree → use that label
- One says Neutral → use the other tool's label
- They conflict → `'Neutral'`

In [ ]:
# ── Build the consensus label ─────────────────────────────────────
def consensus(row):
    v = row['vader_label']
    t = row['tb_label']

    # Case 1: both agree
    if v == t:
        return ____   # Hint: return v

    # Case 2: VADER is Neutral — trust TextBlob
    if v == 'Neutral':
        return ____   # Hint: return t

    # Case 3: TextBlob is Neutral — trust VADER
    if t == 'Neutral':
        return ____   # Hint: return v

    # Case 4: conflict
    return 'Neutral'


df['sentiment'] = df.apply(consensus, axis=1)

print("Consensus sentiment distribution:")
print(df['sentiment'].value_counts())

---
## 🏷️ Task 5 — Classify into 4 Business Groups

### What to do
Combine `loan_status` and `sentiment` to create a `customer_group` column with 4 labels:

| Condition | Group Label |
|---|---|
| Approved + Positive | `'Approved — Positive'` |
| Approved + Negative or Neutral | `'Approved — Negative'` |
| Rejected + Positive | `'Rejected — Positive'` |
| Rejected + Negative or Neutral | `'Rejected — Negative'` |

### Why is this useful?
The bank can use these groups to take different actions:  
Approved-Positive → send referral offer | Rejected-Negative → priority callback

---
## 📊 Task 6 — Visualise Results

### What to do
Create three charts:
1. **Bar chart** — count of customers in each of the 4 groups
2. **Word cloud** — most common words in negative feedback
3. **Scatter plot** — VADER compound vs TextBlob polarity per customer

### Colour guide for bar chart
| Group | Colour |
|---|---|
| Approved — Positive | `'#2ecc71'` (green) |
| Approved — Negative | `'#f39c12'` (orange) |
| Rejected — Positive | `'#3498db'` (blue) |
| Rejected — Negative | `'#e74c3c'` (red) |

In [ ]:
# ── Chart 1: Bar chart of customer groups ────────────────────────
# Step 1: Count customers per group
# Hint: df['customer_group'].value_counts()
group_counts = ____

color_map = {
    'Approved — Positive': '#2ecc71',
    'Approved — Negative': '#f39c12',
    'Rejected — Positive': '#3498db',
    'Rejected — Negative': '#e74c3c'
}
bar_colors = [color_map.get(g, '#95a5a6') for g in group_counts.index]

# Step 2: Draw the bar chart
# Hint: plt.bar(x_values, y_values, color=bar_colors)


In [ ]:
# ── Chart 2: Word cloud of negative feedback ─────────────────────
# Step 1: Filter DataFrame to only Negative sentiment rows
# Hint: df[df['sentiment'] == 'Negative']

# Step 2: Flatten tokens from negative_df into one string
# Hint: ' '.join([token for token_list in negative_df['tokens'] for token in token_list])


# Step 3: Generate and display word cloud


In [ ]:
# ── Chart 3: VADER vs TextBlob scatter ───────────────────────────
sentiment_colors = {'Positive': '#2ecc71', 'Neutral': '#95a5a6', 'Negative': '#e74c3c'}


---
## ✅ Self-Check — Answer These Questions

After completing all 6 tasks, answer the following in the cell below:

1. How many customers were in the **Rejected — Negative** group?
2. What were the **top 3 words** in negative feedback?
3. Did VADER and TextBlob **agree** on more than 70% of customers?
4. Which group had the **highest count** in the bar chart?
5. What does a **high subjectivity score** mean for a customer's feedback?

**Your answers:**

1. 
2. 
3. 
4. 
5. 